In [ ]:
!rm -r /data/sets/nuscenes
!rm v1.0-mini*

In [ ]:
!mkdir -p /data/sets/nuscenes  # Make the directory to store the nuScenes dataset in.

!wget https://www.nuscenes.org/data/v1.0-mini.tgz # Download the nuScenes mini split.

!tar -xf v1.0-mini.tgz -C /data/sets/nuscenes  # Uncompress the nuScenes mini split.

!ls /data/sets/nuscenes

In [ ]:
!ls /data/sets/nuscenes/samples
!ls "/data/sets/nuscenes/samples/CAM_FRONT/n015-2018-07-24-11-22-45+0800__CAM_FRONT__1532402927612460.jpg"
!find /data/sets/nuscenes/samples -name "*CAM_FRONT__1532402927612460*"

In [ ]:
!pip install nuscenes-devkit &> /dev/null  # Install nuScenes.
!pip install --upgrade numpy scipy scikit-learn -q
!pip install ultralytics -q

In [ ]:
%matplotlib inline
from nuscenes.nuscenes import NuScenes

nusc = NuScenes(version='v1.0-mini', dataroot='/data/sets/nuscenes', verbose=True)

### check the classes

In [ ]:
nusc.list_categories()

In [ ]:
names = [cat['name'] for cat in nusc.category]
names

In [ ]:
!pip install ultralytics -q

In [ ]:
from ultralytics import YOLO

# yolo26n.pt
# yolo26s.pt
# yolo26m.pt
# yolo26l.pt
# yolo26x.pt

model = YOLO('yolo26m.pt')
model.names

### try to address the overlapping bbox becauase the bboxs were set in the 3D world

In [ ]:
my_scene = nusc.scene[0]
first_sample_token = my_scene['first_sample_token']
my_sample = nusc.get('sample', first_sample_token)
my_sample = nusc.get('sample', first_sample_token)
for i in range(len(my_sample['anns']) - 1):
    my_annotation_token = my_sample['anns'][i]
    my_annotation_metadata = nusc.get('sample_annotation', my_annotation_token)

    if my_annotation_metadata['visibility_token'] == '1':
        # Get the parent sample
        sample = nusc.get('sample', my_annotation_metadata['sample_token'])

        # Get CAM_FRONT sample_data token directly from the sample
        cam_front_token = sample['data']['CAM_FRONT']
        cam_front_data = nusc.get('sample_data', cam_front_token)

        print(f"Category: {my_annotation_metadata['category_name']}")
        print(f"CAM_FRONT file: {cam_front_data['filename']}")
        print('--------------')

        # nusc.render_sample_data(
        #     cam_front_token,
        #     with_anns=True,
        # )

        # nusc.render_annotation(
        #     my_annotation_token,
        #     extra_info=True
        # )

### further preprocessing

In [ ]:
import os
import shutil
import random
import numpy as np
from PIL import Image
from nuscenes.utils.geometry_utils import view_points

CLASS_MAP = {
    'human.pedestrian.adult':               0,
    'human.pedestrian.child':               0,
    'human.pedestrian.construction_worker': 0,
    'human.pedestrian.police_officer':      0,
    'human.pedestrian.wheelchair':          0,
    'human.pedestrian.stroller':            0,
    'human.pedestrian.personal_mobility':   0,
    'vehicle.bicycle':                      1,
    'vehicle.car':                          2,
    'vehicle.emergency.police':             2,
    'vehicle.motorcycle':                   3,
    'vehicle.bus.bendy':                    4,
    'vehicle.bus.rigid':                    4,
    'vehicle.truck':                        4,
    'vehicle.trailer':                      4,
    'vehicle.construction':                 4,
    'vehicle.emergency.ambulance':          4,
    'movable_object.trafficcone':           5,
    'movable_object.barrier':               5,
}

MIN_VISIBILITY = 2  # skip annotations that are less than 40% visible

CAMERAS =[
    'CAM_FRONT', 'CAM_FRONT_LEFT', 'CAM_FRONT_RIGHT',
    'CAM_BACK_LEFT', 'CAM_BACK_RIGHT', 'CAM_BACK',
]

BASE_DIR = './yolo_dataset'
for split in ['train', 'val']:
    os.makedirs(os.path.join(BASE_DIR, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(BASE_DIR, 'labels', split), exist_ok=True)

# We split by scenes to prevent data leakage between adjacent frames.
scene_tokens = [scene['token'] for scene in nusc.scene]
random.seed(42) # Set seed for reproducible splits
random.shuffle(scene_tokens)

split_idx = int(len(scene_tokens) * 0.8)
train_scenes = set(scene_tokens[:split_idx])
val_scenes = set(scene_tokens[split_idx:])

print(f"Splitting dataset: {len(train_scenes)} Train scenes, {len(val_scenes)} Val scenes.")

# Build visibility lookup once for the whole dataset
ann_visibility = {}
for ann in nusc.sample_annotation:
    try:
        vis = int(ann['visibility_token'])
    except (ValueError, KeyError):
        vis = 4
    ann_visibility[ann['token']] = vis

# Sanity check: print class distribution BEFORE writing any files
yaml_names = {0:'pedestrian',1:'bicycle',2:'car',3:'motorcycle',
               4:'large_vehicle',5:'road_obstacle'}

stats = {
    'total': 0, 'kept': 0,
    'skipped_class': 0, 'skipped_visibility': 0,
    'skipped_depth': 0, 'skipped_bounds': 0
}
class_counts = {i: 0 for i in range(6)}

print("\nExtracting annotations from all 6 cameras...")

for sample in nusc.sample:
    # Determine which split this sample belongs to
    split = 'train' if sample['scene_token'] in train_scenes else 'val'

    out_img_dir = os.path.join(BASE_DIR, 'images', split)
    out_lbl_dir = os.path.join(BASE_DIR, 'labels', split)

    for cam_name in CAMERAS:
        if cam_name not in sample['data']:
            continue

        cam_token = sample['data'][cam_name]
        data_path, boxes, camera_intrinsic = nusc.get_sample_data(cam_token)

        img = Image.open(data_path)
        img_width, img_height = img.size

        label_lines = []

        for box in boxes:
            stats['total'] += 1

            if box.name not in CLASS_MAP:
                stats['skipped_class'] += 1
                continue

            depth = (camera_intrinsic @ box.center.reshape(3, 1))[2, 0]
            if depth < 1.0:
                stats['skipped_depth'] += 1
                continue

            if ann_visibility.get(box.token, 4) < MIN_VISIBILITY:
                stats['skipped_visibility'] += 1
                continue

            corners_2d = view_points(
                box.corners(), camera_intrinsic, normalize=True
            )[:2, :]

            min_x, max_x = np.min(corners_2d[0]), np.max(corners_2d[0])
            min_y, max_y = np.min(corners_2d[1]), np.max(corners_2d[1])

            if (max_x - min_x) > img_width * 1.5:
                stats['skipped_depth'] += 1
                continue

            min_x = max(0, min_x);  max_x = min(img_width,  max_x)
            min_y = max(0, min_y);  max_y = min(img_height, max_y)

            if max_x <= min_x or max_y <= min_y:
                stats['skipped_bounds'] += 1
                continue

            class_id = CLASS_MAP[box.name]
            box_w = max_x - min_x
            box_h = max_y - min_y
            cx    = (min_x + box_w / 2) / img_width
            cy    = (min_y + box_h / 2) / img_height
            nw    = box_w / img_width
            nh    = box_h / img_height

            label_lines.append(
                f"{class_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}"
            )

            # Only count stats for the train split to reflect what the model learns from
            if split == 'train':
                stats['kept'] += 1
                class_counts[class_id] += 1

        if label_lines:
            stem = f"{cam_token}_{cam_name}"
            with open(os.path.join(out_lbl_dir, f"{stem}.txt"), 'w') as f:
                f.write('\n'.join(label_lines))
            shutil.copy(data_path, os.path.join(out_img_dir, f"{stem}.jpg"))

print(f"\n{'='*50}")
print("  TRAIN SPLIT STATS (Excludes Val)")
print(f"  Total boxes seen:          {stats['total']}")
print(f"  Kept:                      {stats['kept']}")
print(f"  Skipped (wrong class):     {stats['skipped_class']}")
print(f"  Skipped (occluded <{MIN_VISIBILITY}):   {stats['skipped_visibility']}")
print(f"  Skipped (bad depth):       {stats['skipped_depth']}")
print(f"  Skipped (out of frame):    {stats['skipped_bounds']}")
print(f"\nPer-class label counts (Train only):")
for cid, cname in yaml_names.items():
    bar = '█' * (class_counts[cid] // 50)
    print(f"  {cid}: {cname:15s} {class_counts[cid]:5d}  {bar}")

In [ ]:
import random
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

#  Config
IMAGES_DIR = './yolo_dataset/images/train'
LABELS_DIR = './yolo_dataset/labels/train'
NUM_SAMPLES = 6

CLASS_NAMES = {
    0: 'pedestrian', 1: 'bicycle', 2: 'car',
    3: 'motorcycle', 4: 'large_vehicle', 5: 'road_obstacle',
}

CLASS_COLORS = {
    0: '#3399FF', 1: '#33D975', 2: '#FFBF1A',
    3: '#FF5733', 4: '#B033FF', 5: '#FF3F8E',
}

#  Helper
def parse_label(label_path, W, H):
    boxes = []
    if not label_path.exists():
        return boxes

    for line in label_path.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) != 5:
            continue

        cid, cx, cy, nw, nh = int(parts[0]), *map(float, parts[1:])
        bw = nw * W
        bh = nh * H
        x0 = cx * W - bw / 2
        y0 = cy * H - bh / 2

        boxes.append(dict(cid=cid, x0=x0, y0=y0, bw=bw, bh=bh))

    return boxes

#  Load images
all_images = sorted(Path(IMAGES_DIR).glob('*.jpg'))
random.seed(11) # make it reproducible
samples = random.sample(all_images, min(NUM_SAMPLES, len(all_images)))

#  Plot
cols = 3
rows = (len(samples) + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 4))
axes = axes.flatten()

for ax, img_path in zip(axes, samples):
    img = Image.open(img_path).convert('RGB')
    W, H = img.size

    label_path = Path(LABELS_DIR) / f"{img_path.stem}.txt"
    boxes = parse_label(label_path, W, H)

    ax.imshow(img)

    # Draw bounding boxes (same method)
    for b in boxes:
        color = CLASS_COLORS[b['cid']]
        rect = patches.Rectangle(
            (b['x0'], b['y0']),
            b['bw'], b['bh'],
            linewidth=1.8,
            edgecolor=color,
            facecolor=color,
            alpha=0.12
        )
        ax.add_patch(rect)

        ax.text(
            b['x0'], b['y0'] - 5,
            CLASS_NAMES[b['cid']],
            fontsize=7,
            color='white',
            bbox=dict(facecolor=color, edgecolor='none', alpha=0.8, pad=1)
        )

    # Show full image path (important for debugging)
    ax.set_title(str(img_path), fontsize=8)

    ax.axis('off')

# Hide empty axes
for ax in axes[len(samples):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

#### check some corrupted annotation

In [ ]:
CAM_TOKEN = "5f37f8e5715b48a7ab08dcf546c90e42"
sample_data_record = nusc.get('sample_data', CAM_TOKEN)
sample_data_record
nusc.render_sample_data(
    CAM_TOKEN,
    with_anns=True,
)

In [ ]:
# Get the sample token from the sample_data_record
sample_token = sample_data_record['sample_token']

# Get the sample record
my_sample = nusc.get('sample', sample_token)

print(f"Annotations for CAM_TOKEN: {CAM_TOKEN}\n")

# Iterate through all annotation tokens in this sample
for ann_token in my_sample['anns']:
    annotation = nusc.get('sample_annotation', ann_token)
    if "human" in annotation['category_name'] and int(annotation['visibility_token']) == 1:
        print(f"Category: {annotation['category_name']}")
        print(f"  Translation (XYZ): {annotation['translation']}")
        print(f"  Size (Width, Length, Height): {annotation['size']}")
        print(f"  Visibility: {annotation['visibility_token']} (1=very low, 4=very high)")
        print("--------------------------------------------------")
        nusc.render_annotation(
            ann_token,
            extra_info=True
        )

### training

In [ ]:
%%writefile model.yaml
path: ./yolo_dataset
train: images/train
val: images/val

nc: 6  # number of classes

names:
  0: pedestrian
  1: bicycle
  2: car
  3: motorcycle
  4: large_vehicle
  5: road_obstacle

In [ ]:
from ultralytics import YOLO

# Load model
model = YOLO('yolo26s.pt')

# Train with Autonomous Driving-tailored augmentations
results = model.train(
    data='model.yaml',
    epochs=50,
    imgsz=640,
    batch=40,
    project='nuscenes_yolo',
    name='tune_run_aug',

    mosaic=1.0,        # Always use mosaic for detecting small objects (pedestrians)
    mixup=0.1,         # Light mixup for robustness
    fliplr=0.5,        # 50% chance to mirror left/right (Valid for roads)
    flipud=0.0,        # NEVER flip upside down for driving data
    degrees=5.0,       # Small rotation to simulate road tilt/bumps
    translate=0.1,     # Slight shifts to simulate panning
    scale=0.5,         # +/- 50% zoom to simulate approaching/leaving objects
    shear=0.0,         # Keep 0, standard driving cams don't shear heavily
    perspective=0.001, # Tiny bit of 3D distortion

    # Color augmentations (Simulate weather/time of day)
    hsv_h=0.015,       # Slight color shifts
    hsv_s=0.7,         # Saturation (helps with gray/cloudy vs sunny days)
    hsv_v=0.4,         # Brightness (helps model handle night/shadows)

    # Occlusion
    erasing=0.4        # Randomly mask out parts of the image (simulates blocked views)
)

#### move the model to colab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil
src = "/content/runs/detect/nuscenes_yolo/tune_run_aug-4"
dst = "/content/drive/MyDrive/yolo_results/tune_run_aug-4"
shutil.copytree(src, dst, dirs_exist_ok=True)

##### check the output size of the model

In [ ]:
import shutil
src = "/content/drive/MyDrive/yolo_results/tune_run_aug-4/weights/best.pt"
dst = "/content/best.pt"
shutil.copy(src, dst)

In [ ]:
import torch
from ultralytics import YOLO

# Load the specific model
model = YOLO('best.pt')

# Ensure model is on the right device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

# 1. Get model parameters
num_classes = len(model.names)

# 2. Run a dummy inference to get the exact output shape
# Input size (1, 3, 640, 640)
dummy_input = torch.zeros(1, 3, 640, 640).to(device)

# Use torch.no_grad() for inference to save memory and avoid gradient errors
model.model.eval()
with torch.no_grad():
    # We call the underlying torch module directly to get the raw tensor
    raw_output = model.model(dummy_input)

# The raw output is often a tuple (inference, loss/metadata) or a list
if isinstance(raw_output, (list, tuple)):
    raw_shape = raw_output[0].shape
else:
    raw_shape = raw_output.shape

# 3. Calculate bytes
# sizeof(float) is 4 bytes
num_detections = raw_shape[2]
output_bytes = 4 * (4 + num_classes) * num_detections

print(f"Model: yolo26s (loaded as best.pt)")
print(f"Device: {device}")
print(f"Number of Classes: {num_classes}")
print(f"Number of Detections: {num_detections}")
print(f"Raw Output Shape: {list(raw_shape)}")
print(f"Output Size in Bytes: {output_bytes:,} bytes ({output_bytes / 1024 / 1024:.2f} MB)")

### validate the model

In [ ]:
from ultralytics import YOLO

model = YOLO('best.pt')

metrics = model.val(
    data='model.yaml',
    split='val',
    plots=True,            # This forces YOLO to generate the F1 and PR curves
    project='nuscenes_val',
    name='evaluation'
)

print(f"mAP50: {metrics.box.map50}")
print(f"mAP50-95: {metrics.box.map}")